# Test: Torque / Speed Controller (UDP → Lua LLC)

This notebook exercises `TorqueSpeedController`, a notebook-friendly class that:
- queries `get_manager_config` (via `execute_request`) to discover UDP endpoints
- sends torque / wheel-speed / vehicle-speed commands over UDP to BeamNG Lua `controller_nn_torque.lua`
- optionally subscribes to `/<vehicle>/reduced_state` (from `gt_state_bridge`) and caches the latest state

## Prereqs
- BeamNG + `sim_manager_node` running
- A scenario loaded with `LowLevelController` configured
- If you want state feedback here: run `gt_state_bridge` so `/<vehicle>/reduced_state` is published


In [2]:
import time

import rclpy

from bng_controller import TorqueSpeedController

# Pick your vehicle name as it appears in the scenario YAML (e.g., 'ego')
VEHICLE = 'EGO'

if not rclpy.ok():
    rclpy.init()

ctl = TorqueSpeedController(
    vehicle_name=VEHICLE,
    subscribe_reduced_state=True,
    spin_in_thread=True,
)

print('Endpoints:', ctl.endpoints)

Endpoints: LowLevelControllerEndpoints(listen_ip='127.0.0.1', listen_port=64257, send_ip='127.0.0.1', send_port=64258)


[INFO] [1769579789.415833878] [torque_speed_controller_EGO]: Subscribed to /EGO/reduced_state
[INFO] [1769579789.418529478] [torque_speed_controller_EGO]: Ready for vehicle='EGO', command_udp=('127.0.0.1', 64257)


In [3]:
# Wait until reduced_state is flowing (requires gt_state_bridge running)

def wait_for_state(timeout_sec: float = 3.0):
    t0 = time.time()
    while time.time() - t0 < timeout_sec:
        st = ctl.get_latest_state_dict()
        if st is not None:
            return st
        time.sleep(0.05)
    return None

state = wait_for_state(3.0)
print('State received:', state is not None)
print('State:', state)

State received: True
State: {'t': 251.36401193915, 'x': 0.23148573828369, 'y': 0.66387901822277, 'yaw': -1.5709409137446, 'V': 0.00031663576591573, 'vx': 0.00029212290323557, 'vy': -0.00012215734796671, 'beta': 0.0, 'r': -0.00014710057379671, 'delta': 0.0, 'wr': -0.0015909715853381, 'wf': 0.00013107304995811, 'we': 144.45489448601, 'pb': 0.0, 'throttle': 0.0, 'brake': 0.0, 'accel_x': -0.40559267486531, 'accel_y': 0.12298996941667}


In [4]:
# Helpers: keep sending commands so the Lua controller doesn't time out

def hold_torque(torque_nm: float, duration_sec: float, rate_hz: float = 50.0):
    dt = 1.0 / rate_hz
    t0 = time.time()
    while time.time() - t0 < duration_sec:
        ctl.command_torque(torque_nm)
        time.sleep(dt)

def hold_wheel_speed(wheel_speed_ms: float, duration_sec: float, rate_hz: float = 50.0):
    dt = 1.0 / rate_hz
    t0 = time.time()
    while time.time() - t0 < duration_sec:
        ctl.command_wheel_speed(wheel_speed_ms)
        time.sleep(dt)

def hold_vehicle_speed(vehicle_speed_ms: float, duration_sec: float, rate_hz: float = 50.0):
    dt = 1.0 / rate_hz
    t0 = time.time()
    while time.time() - t0 < duration_sec:
        ctl.command_vehicle_speed(vehicle_speed_ms)
        time.sleep(dt)

In [8]:
# Test 1: Apply a constant torque for 2 seconds

hold_torque(torque_nm=100.0, duration_sec=2.0)

print('Last command:', ctl.get_last_command())
print('Latest state age (sec):', ctl.get_state_age_sec())
print('Latest state:', ctl.get_latest_state_dict())

Last command: {'wall_time': 1769580211.3899338, 'payload': {'torque': 100.0}}
Latest state age (sec): 0.008104562759399414
Latest state: {'t': 646.74403071869, 'x': -445.82855922819, 'y': -1369.4583979271, 'yaw': -0.82032900021402, 'V': 1.8408273519828, 'vx': 1.8408200135919, 'vy': 0.0051978233296171, 'beta': 0.0028236380945812, 'r': -0.0038912588382871, 'delta': 0.0, 'wr': 1.8326059348858, 'wf': 1.7869240039625, 'we': 212.28794280243, 'pb': -3.4999999999998, 'throttle': 0.052207794040442, 'brake': 0.0, 'accel_x': 0.14685215042165, 'accel_y': 0.0076686054694673}


In [10]:
# Test 2: Command a wheel speed for 3 seconds

hold_wheel_speed(wheel_speed_ms=10.0, duration_sec=100.0)

print('Last command:', ctl.get_last_command())
print('Latest state:', ctl.get_latest_state_dict())

Last command: {'wall_time': 1769580400.033259, 'payload': {'wheel_speed': 10.0}}
Latest state: {'t': 833.27403957839, 'x': -793.34155271969, 'y': -2382.2643454987, 'yaw': -2.0556786087311, 'V': 11.348169749214, 'vx': 11.348168589839, 'vy': 0.0051296753814197, 'beta': 0.00045202668531324, 'r': 0.0012827324878108, 'delta': 0.022119911268095, 'wr': 11.225498973843, 'wf': 11.19597831003, 'we': 516.97587115835, 'pb': -3.3702340126036, 'throttle': 0.13433746766906, 'brake': 0.0, 'accel_x': -0.24745275515508, 'accel_y': -0.062032494771288}


In [ ]:
# Test 3: Command a vehicle speed for 3 seconds (if supported by your Lua controller)

hold_vehicle_speed(vehicle_speed_ms=12.0, duration_sec=3.0)

print('Last command:', ctl.get_last_command())
print('Latest state:', ctl.get_latest_state_dict())

In [ ]:
# Cleanup

ctl.clear_targets()
ctl.stop_spin()
ctl.destroy_node()

# If this notebook is the only ROS user in this kernel, you can shutdown:
# rclpy.shutdown()
print('Done')